# 🧩 Mini-Lab: Git Init, Ignore, Commit & Tag

**Module: Escape the Notebook: Environment, Repo, Git** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Use** Git to initialize a repository, write a `.gitignore`, commit, and tag known-good checkpoints
2. **Explain** why `.gitignore` must be written *before* the first commit
3. **Keep** secrets out of the repository using `.gitignore` and `.env` files
4. **Recover** correctly when a secret is accidentally committed

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Version Control Basics | Init a repo, ignore generated files, commit small changes, and tag v0 to v5 as known good checkpoints you can reset to safely. |
| Secrets Hygiene | Secrets hygiene is the habit of keeping passwords, API keys, and tokens out of code and logs, sharing them safely, and rotating them regularly so one mistake doesn’t become a breach. |


## Why Git for AI Projects?

Recall from **mini-repo-reproducibility-audit**: a reproducible repo lets a stranger clone and run your project. Git is the tool that makes sharing, versioning, and collaborating on that repo possible.

But Git has a dangerous property: **it remembers everything**. Once a file is committed, it lives in the history — even if you delete it later. This is great for code, but catastrophic for secrets like API keys.

The golden rule:

> **Write your `.gitignore` before your first commit.** Anything committed once is in the history forever (unless you rewrite it, which is painful).

## The Git Workflow for a New Project

Every command below runs **in the terminal**. We'll walk through each step, then visualize what happens at each stage.

The sequence matters — follow it in this exact order:

1. `git init` — create the repository
2. Create `.gitignore` — tell Git what to never track
3. Create `.env` — store secrets outside of code
4. `git add` + `git commit` — save a snapshot
5. `git tag` — label the snapshot as a known-good checkpoint

## Step 1 — Initialize the Repository

> 🧩 **version control basics.** Init a repo, ignore generated files, commit small changes, and tag known-good checkpoints.
> Watch for `.git/` after `git init` — nothing is tracked yet.

Navigate to your project folder (the one with `.venv/`, `src/`, etc.) and run:

```bash
git init
```

This creates a hidden `.git/` directory that stores all version history. Your project folder is now a Git repository — but nothing is tracked yet.


## Step 2 — Create `.gitignore` (Before Anything Else)

**This step must happen before `git add`.** The `.gitignore` file tells Git which files and directories to pretend don't exist.

Create a file called `.gitignore` in the project root with the following content:

```
# Virtual environment
.venv/

# Secrets
.env

# Python artifacts
__pycache__/
*.pyc

# IDE settings
.vscode/
.idea/

# OS files
.DS_Store
Thumbs.db
```

Let's look at *why* each entry matters:

In [3]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


gitignore_explained = """
### What Each `.gitignore` Entry Protects Against

| Pattern | What It Ignores | Why |
|---------|----------------|-----|
| `.venv/` | Virtual environment | Hundreds of MBs of binaries; the newcomer recreates it from `requirements.txt` |
| `.env` | Environment variables file | **Contains API keys and secrets** — never commit this |
| `__pycache__/`, `*.pyc` | Python bytecode cache | Auto-generated; differs per OS and Python version |
| `.vscode/`, `.idea/` | IDE settings | Personal preferences; not part of the project |
| `.DS_Store`, `Thumbs.db` | OS-generated metadata | macOS and Windows junk files |

> **The most important line is `.env`** — this is where your API keys live, and it must *never* enter version control.
"""

md(gitignore_explained)


### What Each `.gitignore` Entry Protects Against

| Pattern | What It Ignores | Why |
|---------|----------------|-----|
| `.venv/` | Virtual environment | Hundreds of MBs of binaries; the newcomer recreates it from `requirements.txt` |
| `.env` | Environment variables file | **Contains API keys and secrets** — never commit this |
| `__pycache__/`, `*.pyc` | Python bytecode cache | Auto-generated; differs per OS and Python version |
| `.vscode/`, `.idea/` | IDE settings | Personal preferences; not part of the project |
| `.DS_Store`, `Thumbs.db` | OS-generated metadata | macOS and Windows junk files |

> **The most important line is `.env`** — this is where your API keys live, and it must *never* enter version control.


## Step 3 — Create `.env` for Secrets

Now create a `.env` file in the project root to hold your API keys:

```
OPENAI_API_KEY=sk-your-key-here
```

Because `.env` is already listed in `.gitignore` (from Step 2), Git will ignore it completely. Your code loads secrets from `.env` at runtime using `python-dotenv` — the key never appears in your source files.

### Verify Git ignores it

After creating both files, run:

```bash
git status
```

You should see `.gitignore` listed as an untracked file, but **`.env` should NOT appear at all**. That's the `.gitignore` doing its job.

## Step 4 — Stage and Commit

Now you're ready to make your first commit. This is a two-step process:

### 4a — Stage files

```bash
git add .
```

This stages (prepares) all untracked and modified files — *except* those matched by `.gitignore`. You can verify what's staged:

```bash
git status
```

You should see your project files (`.gitignore`, `requirements.txt`, `src/`, etc.) listed as "Changes to be committed", and `.env` and `.venv/` should be absent.

### 4b — Commit

```bash
git commit -m "Initial commit: project structure with pinned deps"
```

This saves a permanent snapshot of your project. The `-m` flag provides a short message describing what this snapshot contains.

## Step 5 — Tag a Known-Good Checkpoint

Tags let you label specific commits so you can return to them later. After your first commit (and after verifying the project works), tag it:

```bash
git tag v0.1
```

Later, if something breaks, you can always return to this checkpoint:

```bash
git checkout v0.1
```

Tags are especially useful in AI projects where prompt changes or model swaps can break things unexpectedly.

In [ ]:
# Visualize the full workflow as a sequence

workflow = """
### The Complete First-Commit Workflow

```
1. git init                              ← create the repo
2. Create .gitignore  (with .env, .venv/) ← BEFORE any git add
3. Create .env        (API keys here)     ← Git will ignore it
4. git add .                              ← stage everything not ignored
5. git status                             ← verify .env is NOT listed
6. git commit -m "Initial commit"         ← save the snapshot
7. git tag v0.1                           ← label this as known-good
```

> **Order matters.** Steps 2 and 3 must happen before step 4. If you `git add .` before creating `.gitignore`, Git stages `.env` — and your secret enters the history.
"""

md(workflow)

## Secrets Hygiene: What If a Secret Gets Committed?

Mistakes happen. You might forget to add `.env` to `.gitignore`, or accidentally hardcode a key in a Python file. Here's the hard truth and the recovery plan.

In [ ]:
recovery_plan = """
### The Hard Truth About Git and Secrets

| Situation | Severity | What Happened |
|-----------|----------|---------------|
| Secret in `.env`, `.gitignore` has `.env` | ✅ Safe | Git never tracked it |
| Secret committed but **not pushed** | ⚠️ Fixable locally | It's in local history only |
| Secret committed **and pushed** | 🚨 Assume leaked | It's on the remote server's history |

### Recovery Steps (in order)

**Step 1 — Revoke the key immediately**

Go to your API provider's dashboard and rotate/revoke the leaked key. This is always the first step — no amount of Git cleanup changes the fact that the key was exposed.

**Step 2 — Remove the file and recommit**

```bash
# Remove the file from tracking (but keep it on disk)
git rm --cached .env

# Make sure .gitignore has .env
echo ".env" >> .gitignore

# Commit the fix
git commit -m "Remove .env from tracking, add to .gitignore"
```

`git rm --cached` stops tracking the file without deleting it from your disk. The `--cached` flag is important — without it, Git deletes the file.

**Step 3 — If already pushed, scrub history**

Use [`git filter-repo`](https://github.com/newren/git-filter-repo) or [BFG Repo-Cleaner](https://rtyley.github.io/bfg-repo-cleaner/) to rewrite history and remove the secret from all past commits. This is advanced — the key takeaway is that **revoke first, clean up second**.

> **Golden rule:** Revoking the key is the only action that actually stops the damage. History rewriting is cleanup, not protection.
"""

md(recovery_plan)

## Secrets Hygiene Best Practices

Beyond `.gitignore`, here are the habits that keep secrets safe:

In [ ]:
best_practices = """
### Do's and Don'ts

| ✅ Do | ❌ Don't |
|-------|----------|
| Store keys in `.env` | Hardcode keys in Python files |
| Add `.env` to `.gitignore` **before** first commit | Add `.gitignore` after you've already committed secrets |
| Load keys with `os.getenv()` or `python-dotenv` | Print or log API keys (even in debug mode) |
| Provide a `.env.example` with placeholder values | Share `.env` files over Slack, email, etc. |
| Revoke leaked keys immediately | Assume "nobody saw it" |

### The `.env.example` Pattern

Include a `.env.example` in your repo (this **is** committed) so newcomers know what keys they need:

```
# .env.example — copy this to .env and fill in your keys
OPENAI_API_KEY=sk-your-key-here
```

The newcomer copies it to `.env` and fills in their own values. The actual `.env` is ignored by Git.
"""

md(best_practices)

## Quick-Reference Cheat Sheet

In [ ]:
cheat_sheet = """
### Essential Git Commands

| Action | Command |
|--------|---------|
| **Initialize a repo** | `git init` |
| **Check status** | `git status` |
| **Stage all files** | `git add .` |
| **Commit** | `git commit -m "message"` |
| **Tag a checkpoint** | `git tag v0.1` |
| **List tags** | `git tag` |
| **View history** | `git log --oneline` |
| **Return to a tag** | `git checkout v0.1` |
| **Untrack a file (keep on disk)** | `git rm --cached <file>` |

### The Safe Start Sequence

```bash
git init
# Create .gitignore (with .env, .venv/, __pycache__/)
# Create .env (with your API keys)
# Create .env.example (with placeholder values)
git add .
git status          # verify .env is NOT listed
git commit -m "Initial commit"
git tag v0.1
```
"""

md(cheat_sheet)

## 🎯 Summary

### Key Takeaways

1. **`.gitignore` before first commit** — write it before `git add .` so secrets and junk files never enter the history
2. **`.env` for secrets, `.env.example` for documentation** — keep real keys out of the repo while telling newcomers what keys they need
3. **`git tag` marks known-good checkpoints** — label working states so you can return to them when prompt changes or model swaps break things
4. **If a secret leaks, revoke first** — no amount of history rewriting undoes exposure; rotate the key immediately, then clean up the repo
5. **`git rm --cached`** — stops tracking a file without deleting it from disk, essential for removing an accidentally committed `.env`